# Trial {N} — <hypothesis in one line>

**Key insight:** <TBD — fill at the end (also goes into trials.json)>

> ▶ = run the cell · ✍️ = write here before continuing · ⛔ never "Run All" — full rules: `instructions.md`

## 1. ✍️ Read and summarize

Read `context.md` (physics), `trial_00.ipynb` (experiment anchor), **all previous trial
notebooks**, and `trials.json`. Then write below **your current understanding of the
situation**: where the campaign stands, what the last trials showed, what this trial should test.

<agent writes here>

In [ ]:
# 2. ⚙️ Protocol machinery (harness) — frozen for experiment_1 (may change per experiment)
import json, math, os, sys, time
import numpy as np
import torch
torch.set_default_dtype(torch.float32)

# ---- path bootstrap: climb to repo root (dir containing src/) ----
NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, 'src')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, NOTEBOOK_DIR)              # for ag_hypopt
sys.path.insert(0, REPO_ROOT)                 # for src/
os.chdir(REPO_ROOT)                           # data/... resolve from repo root

from src.fitting import nll, fwhm_from_theta, fit_profile
from src.samplers import draw_fixed_noise
from src.implicit import compute_fwhm_and_dgamma

# ============================================================
# 1. PROTOCOL — fixed for the campaign (lives here; changes need Anuar's OK)
# ============================================================
# Benchmark: 14 synthetic experiments, true params from Gregor's fits (16-series).
# Targets are generated at TRUE values with SYNTH_SEED -> identical targets across trials.
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393,   sigma_prop=2.576,  lam=2.232, gamma_true=8.5,  n_target=61),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372,  sigma_prop=3.445,  lam=2.122, gamma_true=8.5,  n_target=358),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316,  sigma_prop=4.141,  lam=2.286, gamma_true=8.5,  n_target=1138),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405,  sigma_prop=7.198,  lam=2.351, gamma_true=8.5,  n_target=2428),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374,  sigma_prop=9.851,  lam=2.593, gamma_true=8.5,  n_target=2424),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365,  sigma_prop=12.627, lam=2.758, gamma_true=8.5,  n_target=2487),
    dict(name='1nW Trans100', power='1nW', mu_true=70.817,  sigma_prop=17.221, lam=2.636, gamma_true=8.5,  n_target=2455),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204,  sigma_prop=3.724,  lam=2.186, gamma_true=14.1, n_target=252),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476,  sigma_prop=5.639,  lam=2.158, gamma_true=14.1, n_target=1572),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279,  sigma_prop=8.319,  lam=2.264, gamma_true=14.1, n_target=2171),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892,  sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95,  lam=2.741, gamma_true=14.1, n_target=2541),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508),
    dict(name='3nW Trans100', power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516),
]

# Reduced benchmark option (open decision): None = full 14; or a list of names.
# Kept as protocol knob so trial_00 can flip it without touching code.
BENCHMARK_SUBSET = None

SYNTH_SEED = 12345    # target generation seed (identical targets every trial)
SEED = 42             # per-step noise seed base (deterministic runs)

# Fixed structural choices (17g/18c verdicts — NOT tunable this campaign):
GAMMA_SCALE = True    # z-form γ-score (bandwidth-normalized, fixed reference)
H_REF = 1.0           # fixed reference bandwidth for the γ-score
LAMBDA_MEAN = 0.0     # mean-matching anchor weight (0.0 = disabled)

# Parallelism (measured: 4 workers @ 17f speed ≈ 160k fit-calls/hour)
N_WORKERS = 4
N_EXP_PARALLEL = 1

# Runtime budget: must make BASELINE_CONFIG (200x200=40k, ~3.5h at measured speed) feasible.
# Full 14-exp benchmark at 17f speed: 40k inner steps x 14 exps / 4 workers ~= 3.5h (18c measured).
# BUDGET_HOURS is a protocol decision — lowering it requires a reduced benchmark or smaller space.
BUDGET_HOURS = 3.5
CALLS_PER_HOUR = 160_000   # total fit calls across all workers, measured 2026-08-29 (18c)

# Baseline config (17g = current best on synthetic): what every trial must beat.
BASELINE_CONFIG = dict(
    n_runs=200, n_iter=200, lr_mu=15.0, lr_gamma=0.5, sigma_ref=10.0,
    clip=10.0, gamma_anneal=0.5, h_s_min=0.05,
)

# ============================================================
# 2. SEARCH SPACE + FEASIBILITY
# ============================================================
def benchmark():
    """Effective benchmark list (full 14 or the configured subset)."""
    if BENCHMARK_SUBSET is None:
        return EXPERIMENTS
    return [e for e in EXPERIMENTS if e['name'] in BENCHMARK_SUBSET]

def runtime_cap(n_exps=None):
    """Max n_runs*n_iter for a ~BUDGET_HOURS trial on the effective benchmark."""
    n_exps = n_exps or len(benchmark())
    return int(BUDGET_HOURS * CALLS_PER_HOUR / n_exps)

def feasible(config, cap=None):
    return config.get('n_runs', 0) * config.get('n_iter', 0) <= (cap if cap is not None else runtime_cap())

# ============================================================
# 4. HARNESS — run one trial + objective
#    NOTE (temporary home, 2026-08-31): this section will move into
#    the template notebook (it may change per experiment).
# ============================================================
def _kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu, sigma_prop, cfg):
    """2D KDE negative log-likelihood + per-data-point scores (identical to 17g)."""
    d_f = data_f[:, None] - sim_f[None, :]
    d_s = data_s[:, None] - sim_s[None, :]
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * (d_s / h_s) ** 2)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / sigma_prop ** 2
    s_mu = (w * score).sum(dim=1)
    if GAMMA_SCALE:
        # z-form: ONE power of bandwidth (unitless distances), fixed reference.
        dlogG = ((d_f * sim_df[None, :]) / h_f + (d_s * sim_ds[None, :]) / h_s) / H_REF
    else:
        dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    nll_val = -logp.mean()
    return s_mu, s_gamma, nll_val, w

def _fit_fn(ph):
    return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)

def _fwhm_fn(th):
    return fwhm_from_theta(th, model='lorentzian')

def _nll_fn(th, ph):
    return nll(th, ph, model='lorentzian', uniform_bg=False)

import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE

def _run_one(args):
    gamma_val, u, b = args
    return compute_fwhm_and_dgamma(gamma_val, u, b, _fit_fn, _fwhm_fn, _nll_fn, n_params=2)

def _init_worker():
    torch.set_num_threads(1)

def _parallel_map(pool, tasks):
    return list(pool.map(_run_one, tasks, chunksize=8))

def _run_experiment(exp, cfg, pool):
    """One experiment: joint μ + γ optimization (17g machinery, config-driven)."""
    mu_true, sigma_prop = exp['mu_true'], exp['sigma_prop']
    lam, gamma_true = exp['lam'], exp['gamma_true']
    mu_init, gamma_init = 0.5 * mu_true, 0.5 * gamma_true
    n_runs = cfg['n_runs']; n_iter = cfg['n_iter']
    lr_mu = cfg['lr_mu']; lr_gamma = cfg['lr_gamma']
    sigma_ref = cfg['sigma_ref']; clip = cfg['clip']
    gamma_anneal = cfg['gamma_anneal']; h_s_min = cfg['h_s_min']

    # ---- synthetic target at TRUE values (same seed as 16-series) ----
    rng_t = np.random.default_rng(SYNTH_SEED)
    tasks_t = []
    for _ in range(exp['n_target']):
        u, b, n = draw_fixed_noise(mu_true, sigma_prop, lam, rng_t)
        tasks_t.append((gamma_true, u.numpy(), b.numpy()))
    res_t = _parallel_map(pool, tasks_t)
    target_f = torch.tensor([r[0] for r in res_t], dtype=torch.float32)
    target_s = torch.tensor([r[1] for r in res_t], dtype=torch.float32)
    scott = exp['n_target'] ** (-1.0 / 6.0)
    H_F = float(target_f.std()) * scott
    H_S = max(float(target_s.std()) * scott, h_s_min)

    mu_val = float(mu_init)
    gamma_val = float(gamma_init)
    history = []
    t_start = time.time()

    for step in range(n_iter):
        rng2 = np.random.default_rng(SEED + step)
        tasks, ns = [], []
        for _ in range(n_runs):
            u, b, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng2)
            tasks.append((gamma_val, u.numpy(), b.numpy()))
            ns.append(n)
        res = _parallel_map(pool, tasks)
        fwhms = [r[0] for r in res]; sigmas = [r[1] for r in res]
        dfs = [r[2] for r in res]; dsigmas = [r[3] for r in res]

        ft = torch.tensor(fwhms, dtype=torch.float32)
        si_t = torch.tensor(sigmas, dtype=torch.float32)
        nt = torch.tensor(ns, dtype=torch.float32)
        dg_t = torch.tensor(dfs, dtype=torch.float32)
        ds_t = torch.tensor(dsigmas, dtype=torch.float32)

        s_mu, s_gamma, nll_val, w = _kde_scores(
            ft, si_t, nt, dg_t, ds_t, target_f, target_s, H_F, H_S, mu_val, sigma_prop, cfg)

        # ---- μ: REINFORCE (σ_ref score, self-normalized, scale-invariant) ----
        B = w.mean(dim=0)
        score = (nt - mu_val) / sigma_ref ** 2
        grad_mu = float(max(min(-(B - B.mean()) @ score, clip), -clip))

        # ---- γ: KDE channel (z-form) ----
        grad_gamma = float(max(min(-s_gamma.mean(), clip), -clip))

        # ---- mean-matching anchor (kernel-free; disabled unless LAMBDA_MEAN > 0) ----
        mean_sim = float(ft.mean()); mean_tgt = float(target_f.mean())
        mean_loss = abs(mean_sim - mean_tgt)
        if LAMBDA_MEAN > 0:
            d_mean_dgamma = float(dg_t.mean())
            grad_gamma_mean = -math.copysign(1.0, mean_sim - mean_tgt) * d_mean_dgamma
            grad_gamma = float(max(min(grad_gamma + LAMBDA_MEAN * grad_gamma_mean, clip), -clip))

        # ---- updates (17f schedules: μ linear decay, no floor; γ anneal) ----
        lr_mu_decay = lr_mu * (1.0 - step / n_iter)
        mu_val -= lr_mu_decay * grad_mu
        mu_val = max(1.0, min(200.0, mu_val))
        gamma_val -= lr_gamma * (1.0 - gamma_anneal * step / n_iter) * grad_gamma
        gamma_val = max(0.1, min(100.0, gamma_val))

        history.append(dict(step=step, mu=mu_val, gamma=gamma_val, nll=float(nll_val),
                            grad_mu=grad_mu, grad_gamma=grad_gamma, mean_loss=mean_loss,
                            mean_n=float(nt.mean())))

    return dict(exp=exp['name'], power=exp['power'],
                mu_true=mu_true, gamma_true=gamma_true,
                mu_final=history[-1]['mu'], gamma_final=history[-1]['gamma'],
                nll_final=history[-1]['nll'], mean_loss_final=history[-1]['mean_loss'],
                history=history, H_F=H_F, H_S=H_S, t_elapsed=time.time() - t_start)

def run_trial(config, experiments=None, verbose=True):
    """Run one trial on the benchmark. Returns dict with per-experiment results.

    config keys (all optional, defaults from BASELINE_CONFIG):
        n_runs, n_iter, lr_mu, lr_gamma, sigma_ref, clip, gamma_anneal, h_s_min
    Deterministic per (config, benchmark): SEED + SYNTH_SEED fixed.
    """
    cfg = dict(BASELINE_CONFIG)
    cfg.update({k: v for k, v in config.items() if v is not None})
    exps = experiments if experiments is not None else benchmark()
    if not feasible(cfg):
        raise ValueError(
            f"infeasible config: n_runs*n_iter={cfg['n_runs']*cfg['n_iter']} "
            f"> runtime_cap={runtime_cap(len(exps))} (~{BUDGET_HOURS}h trial)")

    results = []
    t0 = time.time()
    with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'),
              initializer=_init_worker) as pool:
        for exp in exps:
            if verbose:
                print(f"Running {exp['name']:>12} ...", end=' ', flush=True)
            r = _run_experiment(exp, cfg, pool)
            r['t_elapsed'] = time.time() - t0
            results.append(r)
            if verbose:
                print(f"μ {r['mu_true']:.2f} -> {r['mu_final']:.2f} | "
                      f"γ {r['gamma_true']:.1f} -> {r['gamma_final']:.2f} | "
                      f"NLL {r['nll_final']:.2f}", flush=True)
    total = time.time() - t0
    if verbose:
        print(f"\nTotal: {total/60:.1f} min")
    return dict(config=cfg, benchmark=[e['name'] for e in exps],
                results=results, t_elapsed=total)

def compute_objective(run):
    """Objective = combined relative MSE of (μ, γ) over the benchmark + sampling SE.

    Per experiment: rel-sq error for μ and γ (each normalized by its true value).
    objective = mean over all 2*n_exps errors (lower is better).
    uncertainty = SE of those per-experiment errors (sampling uncertainty across
    experiments — Anuar 2026-08-29 11:15; NO Fisher in the objective).
    Returns (objective, uncertainty, breakdown) — breakdown carries the full
    per-channel MSE/RMSE/rel-RMSE/bias table for reporting.
    """
    results = run['results'] if isinstance(run, dict) else run
    rows = []
    for r in results:
        rel_mu = ((r['mu_final'] - r['mu_true']) / r['mu_true']) ** 2
        rel_g = ((r['gamma_final'] - r['gamma_true']) / r['gamma_true']) ** 2
        rows.append(dict(exp=r['exp'], rel_sq_mu=rel_mu, rel_sq_gamma=rel_g,
                         mu_err=r['mu_final'] - r['mu_true'],
                         gamma_err=r['gamma_final'] - r['gamma_true']))
    errs = np.array([v for row in rows for v in (row['rel_sq_mu'], row['rel_sq_gamma'])])
    objective = float(errs.mean())
    uncertainty = float(errs.std(ddof=1) / np.sqrt(len(errs))) if len(errs) > 1 else 0.0

    mu_e = np.array([row['mu_err'] for row in rows])
    g_e = np.array([row['gamma_err'] for row in rows])
    breakdown = dict(
        n_exps=len(results),
        mu=dict(bias=float(mu_e.mean()), mse=float((mu_e ** 2).mean()),
                rmse=float(np.sqrt((mu_e ** 2).mean())),
                rel_rmse=float(np.sqrt(((mu_e / np.array([r['mu_true'] for r in results])) ** 2).mean()))),
        gamma=dict(bias=float(g_e.mean()), mse=float((g_e ** 2).mean()),
                   rmse=float(np.sqrt((g_e ** 2).mean())),
                   rel_rmse=float(np.sqrt(((g_e / np.array([r['gamma_true'] for r in results])) ** 2).mean()))),
        rows=rows,
    )
    return objective, uncertainty, breakdown

def format_results(breakdown):
    """Compact table for the trial notebook / Telegram (no pandas needed)."""
    lines = [f"{'exp':<12} {'μ_true':>8} {'μ_final':>8} {'Δμ':>8} | "
             f"{'γ_true':>6} {'γ_final':>8} {'Δγ':>8}"]
    for r in breakdown['rows']:
        lines.append(f"{r['exp']:<12} ...")  # detailed rows live in the notebook
    m, g = breakdown['mu'], breakdown['gamma']
    lines.append(f"μ: RMSE {m['rmse']:.3f} (rel {m['rel_rmse']*100:.1f}%) | "
                 f"γ: RMSE {g['rmse']:.3f} (rel {g['rel_rmse']*100:.1f}%)")
    return "\n".join(lines)


In [ ]:
# 3. ▶ Load history + propose 10 candidates (with EI scores)
import json, glob, re, os

TRIALS_PATH = os.path.join(NOTEBOOK_DIR, 'trials.json')
SPACE_PATH = os.path.join(NOTEBOOK_DIR, 'space.json')

# --- structured history ---
trials = json.load(open(TRIALS_PATH))['trials']
best = min((t for t in trials if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print(f"{len(trials)} trials so far | current best: {best['trial_id'] if best else None} "
      f"(obj={best['objective']:.4f})" if best else f"{len(trials)} trials so far | no best yet")

# --- memory index (Key insight of every previous trial; trial_00 first) ---
for p in sorted(glob.glob(os.path.join(NOTEBOOK_DIR, 'trial_*.ipynb'))):
    nb2 = json.load(open(p))
    ki = '?'
    for c in nb2['cells']:
        if c['cell_type'] == 'markdown':
            m = re.search(r'\*\*Key insight:\*\*\s*(.*)', ''.join(c['source']))
            if m: ki = m.group(1).strip()
            break
    tag = '  <-- READ FIRST (experiment anchor)' if os.path.basename(p) == 'trial_00.ipynb' else ''
    print(f'  {os.path.basename(p)}: {ki}{tag}')

# --- TPE: fit on history, then propose 10 candidates ---
from ag_hypopt import AGHyperopt
opt = AGHyperopt(space=SPACE_PATH, feasible=feasible, seed=None)
opt.fit(TRIALS_PATH)
candidates = opt.propose_candidates(n_candidates=10)
print()
print(f"{'ID':>2} | {'EI':>6} | {'explore':>7} | params")
print('-' * 90)
for i, cand in enumerate(candidates, 1):
    cfg = '  '.join(f'{k}={v}' for k, v in cand['params'].items())
    flag = '  <-- exploration' if cand.get('explore') else ''
    print(f'{i:2d} | {cand["ei"]:.4f} | {str(cand.get("explore")):>7} | {cfg}{flag}')

## 4. ✍️ Analyze and choose

Analyze the 10 candidates using **physics reasoning** (context.md failure modes) and the
notebook history. Note any **disagreement with the EI ranking**. **Choose ONE** and justify
in detail: which failure mode it attacks, what you expect to happen, what would confirm or
refute it.

<agent writes here>

In [ ]:
# 5. ▶ Set your choice and run the trial  ⛔ ~3.5h — do not interrupt unless obviously broken
INDEX = 1              # <-- your chosen candidate (1-based, from the table in cell 3)
TRIAL_ID = 'trial_XXX' # <-- next free number

assert 1 <= INDEX <= len(candidates), 'bad INDEX'
CHOSEN = candidates[INDEX - 1]['params']
print('chosen:', CHOSEN)

import traceback, time
t0 = time.time()
try:
    results = run_trial(CHOSEN)
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
    objective, uncertainty = compute_objective(results)
    print(f'objective (MSE vs true) = {objective:.4f} ± {uncertainty:.4f}')
    # TODO: detailed per-experiment results table + plots (convergence, distributions)
except Exception:
    traceback.print_exc()
    results, objective, uncertainty = None, None, None

## 6. ✍️ Analyze the results

Write a comprehensive analysis: what happened vs expectations, **hypothesis confirmed or
refuted** (evidence, not vibes), what was learned about the physics and the hyperparameters,
**ideas worth keeping**, and the **next hypothesis**.

<agent writes here>

In [ ]:
# 7. ▶ Update trials.json — fill the two strings below first, then run
SUMMARY = '<one-line summary of this trial, from your analysis above>'
KEY_INSIGHT = '<one sentence — same as the title cell>'

entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': SUMMARY,
    'key_insight': KEY_INSIGHT,
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)

## 8. ↺ Loop back

Return to `instructions.md` and start the next trial (copy this template to the next number).

If this trial motivates a **structural change** (score, likelihood, model, protocol):
describe it here and **request Anuar's permission** — do NOT run it yourself.